In [7]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# ==============================================================================
# 1. LOAD DATA & STANDARISASI WAKTU
# ==============================================================================
print("Memuat dataset dan standarisasi waktu...")
file_path = 'Database.xlsx'

# Muat data & bersihkan anomali spasi pada nama kolom
df_meta = pd.read_excel(file_path, sheet_name='DB_ID_STOCKS').rename(columns=lambda x: str(x).strip())
df_prices = pd.read_excel(file_path, sheet_name='P_ID_STOCKS').rename(columns=lambda x: str(x).strip())
df_rf = pd.read_excel(file_path, sheet_name='IND_10Y_BOND').rename(columns=lambda x: str(x).strip())

# Standarisasi indeks waktu menjadi Datetime untuk mempermudah perhitungan seri waktu
df_prices['MONTH'] = pd.to_datetime(df_prices['MONTH'])
df_prices = df_prices.sort_values('MONTH').set_index('MONTH')

df_rf['MONTH'] = pd.to_datetime(df_rf.iloc[:, 0])
df_rf = df_rf.sort_values('MONTH').set_index('MONTH')

# Konversi Risk-Free Rate (SBN) ke desimal bulanan (de-compounding)
rf_num = pd.to_numeric(df_rf.iloc[:, 1], errors='coerce')
df_rf['RF_MONTHLY'] = (1.0 + np.where(rf_num > 1.0, rf_num / 100.0, rf_num)) ** (1.0 / 12.0) - 1.0


Memuat dataset dan standarisasi waktu...


In [8]:
# ==============================================================================
# 2. PERSIAPAN UNIVERSE & PEMBENTUKAN FAKTOR FAMA-FRENCH
# ==============================================================================
print("Membentuk 5 Faktor Fama-French (SMB, HML, RMW, CMA)...")
# Saring saham mapan (> 5 tahun) untuk meminimalisir risiko suspensi/delisting
valid_tickers = [t for t in df_meta.loc[df_meta['FG_5_YEARS'] == 1, 'TICKERS'].dropna().unique() if t in df_prices.columns]

# Kalkulasi Imbal Hasil (Return) Aktual Bulanan
df_all_returns = df_prices.pct_change().dropna(how='all')
rf_series = df_rf['RF_MONTHLY'].reindex(df_all_returns.index).ffill().bfill()
mkt_rf_series = df_all_returns['JKSE'] - rf_series

df_meta_clean = df_meta.set_index('TICKERS')
years_avail = sorted([int(c.split('_FY')[1]) for c in df_meta_clean.columns if 'TOTAL_ASSETS_FY' in c])
valid_dates, smb_hist, hml_hist, rmw_hist, cma_hist = [], [], [], [], []

# Ekstrak fundamental historis & susun portofolio faktor (Size, Value, Profitability, Investment)
for dt in df_all_returns.index:
    curr_returns = df_all_returns.loc[dt].drop('JKSE', errors='ignore').dropna()
    avail = curr_returns.index.intersection(df_meta_clean.index)
    if len(avail) < 20: continue
    
    fy_tgt = next((y for y in [dt.year - 1, min(years_avail), max(years_avail)] if y in years_avail), max(years_avail))
    fy_prev = fy_tgt - 1 if (fy_tgt - 1) in years_avail else fy_tgt
    sub = df_meta_clean.loc[avail]
    
    # Ambil matriks fundamental
    mc = pd.to_numeric(sub.get(f'MARKETCAP_FY{fy_tgt}', sub['MARKET_CAP_TODAY']), errors='coerce')
    eq = pd.to_numeric(sub.get(f'TOTAL_COMMON_EQUITY_FY{fy_tgt}'), errors='coerce')
    eq_prev = pd.to_numeric(sub.get(f'TOTAL_COMMON_EQUITY_FY{fy_prev}', eq), errors='coerce')
    ta = pd.to_numeric(sub.get(f'TOTAL_ASSETS_FY{fy_tgt}'), errors='coerce')
    ta_prev = pd.to_numeric(sub.get(f'TOTAL_ASSETS_FY{fy_prev}', ta), errors='coerce')
    
    op_num = (pd.to_numeric(sub.get(f'TOTAL_REV_FY{fy_tgt}', 0), errors='coerce').fillna(0) - 
              pd.to_numeric(sub.get(f'COGS_FY{fy_tgt}', 0), errors='coerce').fillna(0) - 
              pd.to_numeric(sub.get(f'SGA_FY{fy_tgt}', 0), errors='coerce').fillna(0) - 
              pd.to_numeric(sub.get(f'INT_EXP_FY{fy_tgt}', 0), errors='coerce').fillna(0))

    m_df = pd.DataFrame({'RET': curr_returns[avail], 'MC': mc, 'BM': eq / mc, 
                         'OP': op_num / eq_prev, 'INV': (ta - ta_prev) / ta_prev.replace(0, np.nan)}).dropna(subset=['RET', 'MC'])
    
    if len(m_df) < 10: continue
    
    # Kalkulasi selisih return kuartil 30/70 untuk menemukan premi faktor
    med_mc = m_df['MC'].median()
    smb_hist.append(m_df[m_df['MC'] <= med_mc]['RET'].mean() - m_df[m_df['MC'] > med_mc]['RET'].mean())
    
    for factor, lst in zip(['BM', 'OP', 'INV'], [hml_hist, rmw_hist, cma_hist]):
        q30, q70 = m_df[factor].dropna().quantile([0.3, 0.7]) if not m_df[factor].dropna().empty else (0, 0)
        lst.append(m_df[m_df[factor] <= q30]['RET'].mean() - m_df[m_df[factor] >= q70]['RET'].mean() if factor == 'INV' else 
                   m_df[m_df[factor] >= q70]['RET'].mean() - m_df[m_df[factor] <= q30]['RET'].mean())
    valid_dates.append(dt)

df_factors = pd.DataFrame({'MKT_RF': mkt_rf_series.reindex(valid_dates), 'SMB': smb_hist, 'HML': hml_hist, 'RMW': rmw_hist, 'CMA': cma_hist}, index=valid_dates)


Membentuk 5 Faktor Fama-French (SMB, HML, RMW, CMA)...


In [9]:
# ==============================================================================
# 3. REGRESI OLS, SCORING, & PENYUSUNAN METRIK FINAL
# ==============================================================================
print("Menjalankan Regresi Fama-French & Mengkalkulasi Metrik Akhir...")
df_excess = df_all_returns[valid_tickers].sub(rf_series, axis=0)
reg_data = df_excess.join(df_factors, how='inner').dropna()
factor_means = reg_data[['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA']].mean()

X = sm.add_constant(reg_data[['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA']])
metrics_all = {}

# Regresi individu untuk tiap saham tanpa pengulangan loop ganda
for ticker in valid_tickers:
    y = reg_data[ticker]
    model = sm.OLS(y, X).fit(cov_type='HC3') # Parameter HC3 agar kebal terhadap data pencilan (outlier)
    
    hist_returns = df_all_returns[ticker].dropna()
    vol = hist_returns.std()
    mean_tot = hist_returns.mean()
    
    metrics_all[ticker] = {
        'EXPECTED_RETURN': np.dot(model.params.drop('const'), factor_means),
        'VARIANCE': model.resid.var(), 
        'RISK': vol,
        'SHARPE_RATIO': df_excess[ticker].dropna().mean() / vol if vol != 0 else 0,
        'MAX_GAIN': mean_tot + (2 * vol),
        'MAX_LOSS': max(mean_tot - (2 * vol), -1.0) # Limitasi risiko ekstrem agar tidak lebih dari -100%
    }


Menjalankan Regresi Fama-French & Mengkalkulasi Metrik Akhir...


In [10]:
# ==============================================================================
# 4. EXPORT KE CSV (FINALISASI)
# ==============================================================================
df_export = pd.DataFrame.from_dict(metrics_all, orient='index')
df_export.index.name = 'TICKER'

# Sinkronisasi atribut profil perusahaan
df_export['NAME'] = df_export.index.map(df_meta_clean['NAME'])
df_export['MARKET_CAPITALIZATION'] = df_export.index.map(pd.to_numeric(df_meta_clean['MARKET_CAP_TODAY'], errors='coerce'))

# Pembuatan peringkat (Ranking 1 = Terbaik). Berbasis nilai Sharpe tertinggi dan Risiko terendah.
skor = df_export['SHARPE_RATIO'].rank(ascending=False) + df_export['RISK'].rank(ascending=True)
df_export['RANKING'] = skor.rank(ascending=True).astype(int) 

# Pengaturan tata letak kolom
cols_order = ['NAME', 'MARKET_CAPITALIZATION', 'EXPECTED_RETURN', 'RISK', 'SHARPE_RATIO', 'RANKING', 'MAX_GAIN', 'MAX_LOSS', 'VARIANCE']
df_export = df_export[cols_order].sort_values('RANKING')

# Ekspor Data
file_name = 'Hasil_Regresi_Fama_French_5_Faktor.csv'
df_export.to_csv(file_name)

print(f"BERHASIL! Data {len(df_export)} saham telah diekspor ke '{file_name}'.")
display(df_export.head(10))

BERHASIL! Data 649 saham telah diekspor ke 'Hasil_Regresi_Fama_French_5_Faktor.csv'.


,NAME,MARKET_CAPITALIZATION,EXPECTED_RETURN,RISK,SHARPE_RATIO,RANKING,MAX_GAIN,MAX_LOSS,VARIANCE
TICKER,,,,,,,,,
FAPA,PT FAP AGRI Tbk (XIDX:FAPA),26948380000000,0.003932,0.029119,0.408100,1,0.075544,-0.040932,0.000918
BPII,PT Batavia Prosperindo Internasional Tbk (XIDX...,4546711000000,0.005678,0.078104,0.140054,2,0.172866,-0.139549,0.007110
VICI,PT VICTORIA CARE INDONESIA Tbk (XIDX:VICI),6473220000000,0.005760,0.080121,0.147830,3,0.177505,-0.142981,0.005990
DNET,PT Indoritel Makmur Internasional Tbk. (XIDX:D...,137939400000000,-0.003901,0.094663,0.155168,4,0.209733,-0.168917,0.011903
MTDL,PT Metrodata Electronics Tbk (XIDX:MTDL),6113889000000,-0.009577,0.074866,0.100209,5,0.162953,-0.136509,0.004629
SFAN,PT SURYA FAJAR CAPITAL Tbk (XIDX:SFAN),2719868000000,-0.000203,0.100493,0.125160,6,0.219040,-0.182933,0.000639
ADES,PT Akasha Wira International Tbk (XIDX:ADES),20056490000000,0.004434,0.115566,0.199302,7,0.259885,-0.202381,0.014888
MIDI,PT Midi Utama Indonesia Tbk (XIDX:MIDI),8693177000000,-0.007771,0.087895,0.100980,8,0.190386,-0.161195,0.009979
TAPG,PT Triputra Agro Persada Tbk (XIDX:TAPG),40300660000000,0.004965,0.112951,0.157885,9,0.249161,-0.202644,0.013098
